In [1]:
!pip install requests yfinance google-cloud-pubsub

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\Stock_pred\stockanalysis-444512-3fd7af0c6e0a.json"

In [3]:
import yfinance as yf
from google.cloud import pubsub_v1
import json
from datetime import datetime, timezone

tickers = ["TSLA", "AAPL", "MSFT", "GOOGL", "AMZN" ,"ORCL" ,"INTC" ,"NVDA" ,"META" ,"BABA"]  # Add more tickers as needed

data = {}
try:
    for ticker in tickers:
        stock_data = yf.download(ticker, interval='1d', period='3mo')
        if not stock_data.empty:
            data[ticker] = stock_data
        else:
            print(f"No data fetched for {ticker}.")
    print("Stock data fetched successfully.")
except Exception as e:
    print(f"Error fetching stock data: {e}")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Stock data fetched successfully.


In [4]:
# Authenticate and set up Pub/Sub publisher
try:
    publisher = pubsub_v1.PublisherClient()
    print("Google Pub/Sub authentication successful.")
except Exception as e:
    print(f"Authentication failed: {e}")
    raise

Google Pub/Sub authentication successful.


In [5]:
import requests, time
from google.cloud import bigquery

# Pub/Sub setup
project_id = "stockanalysis-444512"
topic_id = "real-time-stock-data"
publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path(project_id, topic_id)

# Publish stock data to Pub/Sub
try:
    for ticker, stock_data in data.items():
        latest_row = stock_data.iloc[-1]  # Get the most recent stock data

        stock_message = {
            "symbol": ticker,
            "price": round(float(latest_row["Close"].iloc[0]), 2),  # Convert to a JSON-compatible float
            "timestamp": latest_row.name.strftime('%Y-%m-%dT%H:%M:%SZ'),  # Format timestamp
        }
        message = json.dumps(stock_message).encode("utf-8")
        future = publisher.publish(topic_path, message)
        print(f"Published message for {ticker} with ID: {future.result()}")
except Exception as e:
    print(f"Error publishing to Pub/Sub: {e}")


Published message for TSLA with ID: 13582636349129429
Published message for AAPL with ID: 13582482284764075
Published message for MSFT with ID: 13582443804119255
Published message for GOOGL with ID: 13582803555359311
Published message for AMZN with ID: 13582803608227381
Published message for ORCL with ID: 13582672760475393
Published message for INTC with ID: 13582503849130962
Published message for NVDA with ID: 13582481189094738
Published message for META with ID: 13582635057048771
Published message for BABA with ID: 13582862095659300


In [ ]:
# Subscriber for Pub/Sub
subscription_id = 'real-time-stock-data-sub'
subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(project_id, subscription_id)

client = bigquery.Client(project="stockanalysis-444512")
table_ref = "stockanalysis-444512.stock_analysis.stream_pubsubtobq"

def callback(message):
    try:
        if not message.data:
            print("Empty message received. Skipping...")
            message.ack()
            return
        
        raw_data = message.data.decode("utf-8")
        print(f"Raw message data: {raw_data}")
        
        data = json.loads(raw_data)
        print(f"Parsed data: {data}")
        
        # Validate keys in the message
        required_keys = {"symbol", "price", "timestamp"}
        if not required_keys.issubset(data.keys()):
            print(f"Missing keys in the message: {data.keys()}")
            message.nack()
            return
        
        # Insert into BigQuery
        row_to_insert = {
            "symbol": data["symbol"],
            "price": data["price"],
            "timestamp": data["timestamp"],
        }
        errors = client.insert_rows_json(table_ref, [row_to_insert])
        if errors:
            print(f"BigQuery errors: {errors}")
        else:
            print(f"Data for {data['symbol']} inserted into BigQuery.")
        message.ack()
    except Exception as e:
        print(f"Unexpected error: {e}")
        message.nack()

# Start the subscriber
subscriber.subscribe(subscription_path, callback=callback)
print(f"Listening for messages on {subscription_path}...")

# Keep the subscriber running
while True:
    time.sleep(60)

Listening for messages on projects/stockanalysis-444512/subscriptions/real-time-stock-data-sub...
Raw message data: {"symbol": "TSLA", "price": 426.5, "timestamp": "2025-01-17T00:00:00Z"}
Parsed data: {'symbol': 'TSLA', 'price': 426.5, 'timestamp': '2025-01-17T00:00:00Z'}
Raw message data: {"symbol": "GOOGL", "price": 196.0, "timestamp": "2025-01-17T00:00:00Z"}
Parsed data: {'symbol': 'GOOGL', 'price': 196.0, 'timestamp': '2025-01-17T00:00:00Z'}
Raw message data: {"symbol": "AMZN", "price": 225.94, "timestamp": "2025-01-17T00:00:00Z"}
Parsed data: {'symbol': 'AMZN', 'price': 225.94, 'timestamp': '2025-01-17T00:00:00Z'}
Raw message data: {"symbol": "NVDA", "price": 137.71, "timestamp": "2025-01-17T00:00:00Z"}
Parsed data: {'symbol': 'NVDA', 'price': 137.71, 'timestamp': '2025-01-17T00:00:00Z'}
Raw message data: {"symbol": "AAPL", "price": 229.98, "timestamp": "2025-01-17T00:00:00Z"}
Parsed data: {'symbol': 'AAPL', 'price': 229.98, 'timestamp': '2025-01-17T00:00:00Z'}
Raw message data: {